# Getting data into LSDB

This notebook presents two modes of loading data into LSDB, in HiPSCat format:

1. The `lsdb.from_dataframe()` method: helpful to load smaller catalogs from a single dataframe. Data should have fewer than 1-2 million rows and the pandas dataframe should be less than 1-2G in-memory. If your data is larger, the format is complicated, you need more flexibility, or you notice any performance issues when importing with this mode, use the next mode.
2. The `hipscat-import` package: for large datasets (1G - 100s of TB). This is a purpose-built map-reduce pipeline for creating hipscat catalogs from various datasets. In this notebook, we use a very basic dataset and basic import options. Please see [the full package documentation](https://hipscat-import.readthedocs.io/) if you need to do anything more complicated.

To demonstrate both methods we will use `small_sky_order1`, a small object catalog contained in a single CSV file.

In [ ]:
import tempfile
from pathlib import Path

In [ ]:
test_data_dir = Path.cwd() / ".." / ".." / "tests" / "data"

# Input paths
catalog_dir = test_data_dir / "small_sky_order1"
catalog_csv_path = catalog_dir / "small_sky_order1.csv"

# Temporary directory for the intermediate/output files
tmp_dir = tempfile.TemporaryDirectory()

## Using `lsdb.from_dataframe()`

In [ ]:
import lsdb
import pandas as pd

In [ ]:
%%time

# Read simple catalog from its CSV file
catalog = lsdb.from_dataframe(pd.read_csv(catalog_csv_path), catalog_name="small_sky_order1")

# Save the catalog to disk in parquet HiPSCat format
catalog.to_hipscat(f"{tmp_dir.name}/from_dataframe/small_sky_order1")

Please see the method's [documentation](https://lsdb.readthedocs.io/en/stable/autoapi/lsdb/loaders/dataframe/from_dataframe/index.html) to see the available keyword arguments for advanced use cases.

## Using `hipscat-import`

Let's install the latest version of `hipscat-import`:

In [ ]:
!pip install git+https://github.com/astronomy-commons/hipscat-import.git@main --quiet

In [ ]:
from dask.distributed import Client
from hipscat_import.catalog.arguments import ImportArguments
from hipscat_import.pipeline import pipeline_with_client

In [ ]:
%%time

args = ImportArguments(
    ra_column="ra",
    dec_column="dec",
    file_reader="csv",
    input_file_list=[catalog_csv_path],
    output_path=f"{tmp_dir.name}/hipscat_import",
    output_artifact_name="small_sky_order1",
    resume=False,
)

with Client(n_workers=1) as client:
    pipeline_with_client(args, client)

Please see the method's [documentation](https://hipscat-import.readthedocs.io/en/stable/autoapi/hipscat_import/catalog/arguments/index.html#hipscat_import.catalog.arguments.ImportArguments) to see the available keyword arguments for advanced use cases.

Let's read both catalogs, from disk, and check that the two methods produced the same output:

In [ ]:
small_sky_from_dataframe = lsdb.read_hipscat(f"{tmp_dir.name}/from_dataframe/small_sky_order1")
small_sky_from_dataframe

In [ ]:
small_sky_hc_import = lsdb.read_hipscat(f"{tmp_dir.name}/hipscat_import/small_sky_order1")
small_sky_hc_import

In [ ]:
# Verify that the pixels they contain are similar
assert small_sky_from_dataframe.get_healpix_pixels() == small_sky_hc_import.get_healpix_pixels()

# Verify that resulting dataframes contain the same data
sorted_from_dataframe = small_sky_from_dataframe.compute().sort_index()
sorted_from_import_pipeline = small_sky_hc_import.compute().sort_index()
pd.testing.assert_frame_equal(sorted_from_dataframe, sorted_from_import_pipeline)

Finally, tear down the directory used for the intermediate / output files:

In [ ]:
tmp_dir.cleanup()